# 06 - Download the public assembly cohort and run the first serious baseline

The four-isolate smoke workflow passed end to end. This notebook downloads the remaining ENA assembled FASTA files and trains the first conventional sequence baseline across the resolved cohort. It samples evenly spaced genomic windows directly from each assembly and does not materialize a full genomic-window CSV.

## 1. Configuration

The download switch defaults to `False`. Review the disk projection before enabling it. The downloader preserves at least 15 GB of free disk space.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ASSEMBLY_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_assembly_retrieval_plan.csv"
SPLITS_PATH = PROJECT_ROOT / "data" / "processed" / "isolate_splits.csv"
ASSEMBLY_DIR = PROJECT_ROOT / "data" / "raw" / "assemblies"
FULL_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
METRICS_PATH = PROJECT_ROOT / "reports" / "full_kmer_assembly_baseline.json"
MINIMUM_FREE_GB = 15.0
DOWNLOAD_FULL_COHORT = False
RUN_FULL_BASELINE = False

print("Project root:", PROJECT_ROOT)

## 2. Review storage before downloading

The conservative output estimate applies a five-times multiplier to ENA's compressed byte total.

In [ ]:
assembly_plan = pd.read_csv(ASSEMBLY_PLAN_PATH, dtype={"label": str}).fillna("")
compressed_gb = assembly_plan["submitted_bytes"].sum() / 1_000_000_000
conservative_output_gb = compressed_gb * 5
free_gb = shutil.disk_usage(PROJECT_ROOT).free / 1_000_000_000

print("Assemblies planned:", len(assembly_plan))
print("Compressed ENA FASTA total: %.2f GB" % compressed_gb)
print("Conservative decompressed estimate: %.2f GB" % conservative_output_gb)
print("Current free disk space: %.2f GB" % free_gb)
print("Required post-download reserve: %.2f GB" % MINIMUM_FREE_GB)

## 3. Download compact public FASTA assemblies

Change `DOWNLOAD_FULL_COHORT` to `True` after reviewing the estimate. Existing four-isolate smoke files are reused.

In [ ]:
download_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "download_ena_assemblies.py"),
    "--plan",
    str(ASSEMBLY_PLAN_PATH),
    "--output-dir",
    str(ASSEMBLY_DIR),
    "--minimum-free-gb",
    str(MINIMUM_FREE_GB),
]
print(" ".join(download_command))
if DOWNLOAD_FULL_COHORT:
    subprocess.run(download_command, cwd=PROJECT_ROOT, check=True)

## 4. Create the assembly-backed manifest

This cell stops if the cohort is incomplete.

In [ ]:
assembly_plan["assembly_path"] = assembly_plan["isolate_id"].map(
    lambda isolate_id: str((ASSEMBLY_DIR / f"{isolate_id}.fasta").relative_to(PROJECT_ROOT))
)
assembly_plan["exists"] = assembly_plan["assembly_path"].map(lambda path: (PROJECT_ROOT / path).exists())
missing = assembly_plan.loc[~assembly_plan["exists"], "isolate_id"].tolist()
print("Available assemblies:", int(assembly_plan["exists"].sum()), "/", len(assembly_plan))
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} assemblies. Enable DOWNLOAD_FULL_COHORT first.")

splits = pd.read_csv(SPLITS_PATH, dtype={"label": str}).fillna("")
full_manifest = splits[splits["isolate_id"].isin(assembly_plan["isolate_id"])].copy()
full_manifest = full_manifest.drop(columns=["assembly_path"]).merge(
    assembly_plan[["isolate_id", "assembly_path"]],
    on="isolate_id",
    how="left",
    validate="one_to_one",
)
full_manifest.to_csv(FULL_MANIFEST_PATH, index=False)
print(pd.crosstab(full_manifest["split"], full_manifest["label"], margins=True))
print("Wrote:", FULL_MANIFEST_PATH.relative_to(PROJECT_ROOT))

## 5. Train the direct-assembly k-mer baseline

Change `RUN_FULL_BASELINE` to `True` only after the complete assembly manifest is written. The direct-assembly baseline samples 64 evenly spaced 999-base windows per isolate to keep the fit comfortable on a laptop.

In [ ]:
baseline_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_kmer_baseline_from_assemblies.py"),
    "--manifest",
    str(FULL_MANIFEST_PATH),
    "--splits",
    str(SPLITS_PATH),
    "--output",
    str(METRICS_PATH),
]
print(" ".join(baseline_command))
if RUN_FULL_BASELINE:
    subprocess.run(baseline_command, cwd=PROJECT_ROOT, check=True)
if METRICS_PATH.exists():
    print(METRICS_PATH.read_text())

## 6. Next step

Review the grouped-test baseline before running DNABERT2 embeddings. The transformer stage should use capped windows per isolate and can be staged on the four-isolate cohort first.